In [ ]:
# Librerías
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Obtener la raíz del proyecto (sube dos niveles desde notebooks/)
PROJECT_ROOT = Path().resolve().parents[1]
# Construir la ruta al archivo
data_path = PROJECT_ROOT / "data" / "raw" / "serie-ena-departamento-cultivos-permanentes-2019-II.xlsx"
print(f"Ruta del archivo: {data_path}")

Ruta del archivo: /home/ddayann/proyectos/Coffe/proyecto_aplicado_en_analitica_de_datos/data/raw/serie-ena-departamento-cultivos-permanentes-2019-II.xlsx


In [3]:
# Lectura de archivo ENA
ena_cafe = pd.read_excel(data_path, sheet_name="CAFÉ", header = 11)
ena_cafe.head(10)

,Código,Nombre,2012,2013,2014,2015,2016,2017,2018,2019,...,Unnamed: 28,2012.3,2013.3,2014.3,2015.3,2016.3,2017.3,2018.3,2019.3,Unnamed: 37
0,NaN,Total Nacional,722110.135470,728530.891460,757744.246488,768140.300583,711010.951995,814808.260000,815103.180000,839660.630000,...,NaN,1.157527,1.511370,1.513615,1.459273,1.444480,1.650000,1.281481,1.291483,NaN
1,NaN,CVE,3.295465,4.152058,3.281192,3.199942,3.620615,5.570000,3.833942,3.743286,...,NaN,3.589880,3.187949,3.717393,3.349207,4.792712,2.550000,3.458122,1.959314,NaN
2,NaN,IC95%±,46641.893547,59288.095141,48731.566195,48176.884729,50456.220179,88954.247361,61251.140000,61604.560000,...,NaN,0.081445,0.094436,0.110283,0.095793,0.135690,0.082467,0.090000,0.050000,NaN
3,NaN,Total Región Andina,553857.125915,573539.867474,584278.942456,595536.937553,534971.814554,596821.660000,595940.690000,611408.630000,...,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.306341,1.310373,NaN
4,NaN,CVE,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.340771,3.846376,...,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.773191,1.913662,NaN
5,NaN,IC95%±,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,50702.110000,46093.470000,...,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.100000,0.050000,NaN
6,05,Antioquia,117793.193165,117782.923762,120224.163164,129080.485754,113596.273219,116156.340000,116370.980000,120224.640000,...,NaN,1.468688,2.210691,2.123736,2.040202,2.343240,2.030000,1.388191,1.477795,NaN
7,NaN,CVE,9.141294,0.000000,9.118823,9.214668,12.091430,13.690000,10.017462,10.668271,...,NaN,7.660369,7.473488,10.284362,9.890785,11.204250,4.010000,6.753650,5.097765,NaN
8,NaN,IC95%±,21104.931901,0.000000,21487.536397,23312.902691,26921.410588,31167.533774,22848.540000,25138.750000,...,NaN,0.220514,0.323823,0.428089,0.395512,0.514583,0.159550,0.180000,0.150000,NaN
9,15,Boyacá,4791.050151,6239.759270,4232.813418,5255.623320,7206.308017,10001.400000,10777.680000,12312.950000,...,NaN,0.478355,0.911421,0.754211,0.645100,1.002199,1.100000,1.000900,0.893929,NaN


In [4]:
ena_cafe = ena_cafe[ena_cafe["Código"].notna()]                 # Elimina filas con código vacío    
ena_cafe = ena_cafe[ena_cafe["Nombre"].notna()]                 # Elimina filas con nombre vacío
ena_cafe = ena_cafe.dropna(axis=1, how='all')                   # Eliminar columnas completamente vacías
# ena_cafe

In [5]:
# Renombrar columnas
years = list(range(2012, 2020))

new_columns = (
    ["Codigo", "Departamento"] +
    [f"Area_plantada_{y}" for y in years] +
    [f"Area_productiva_{y}" for y in years] +
    [f"Produccion_{y}" for y in years] +
    [f"Rendimiento_{y}" for y in years]
)

ena_cafe.columns = new_columns
# ena_cafe.head()

In [6]:
# Tipificación de datos
ena_cafe["Codigo"] = ena_cafe["Codigo"].astype(str).str.zfill(2)                           # Código a texto
ena_cafe["Departamento"] = ena_cafe["Departamento"].astype(str).str.strip()                # Departamento a texto
value_cols = [col for col in ena_cafe.columns if col not in ["Codigo", "Departamento"]]    # Columnas de valores
ena_cafe[value_cols] = ena_cafe[value_cols].apply(pd.to_numeric, errors="coerce")          # Campos numéricos, errores a NaN

In [7]:
# Pasar de formato ancho a largo
bases_long = []

for y in years:
    temp = ena_cafe[[
        "Codigo",
        "Departamento",
        f"Area_plantada_{y}",
        f"Area_productiva_{y}",
        f"Produccion_{y}",
        f"Rendimiento_{y}"
    ]].copy()

    temp.columns = [
        "Codigo",
        "Departamento",
        "Area_plantada",
        "Area_productiva",
        "Produccion",
        "Rendimiento"
    ]

    temp["Año"] = y
    bases_long.append(temp)

ena_cafe_long = pd.concat(bases_long, ignore_index=True)

# Reordenar columnas
ena_cafe = ena_cafe_long[
    ["Codigo", 
    "Departamento",
    "Año",
    "Area_plantada",
    "Area_productiva",
    "Produccion",
    "Rendimiento"]
    ].sort_values(["Año", "Departamento"]).reset_index(drop=True)

In [8]:
ena_cafe

,Codigo,Departamento,Año,Area_plantada,Area_productiva,Produccion,Rendimiento
0,91,Amazonas,2012,0.000000,0.000000,0.00000,0.000000
1,05,Antioquia,2012,117793.193165,71671.225723,105262.70132,1.468688
2,81,Arauca,2012,0.000000,0.000000,0.00000,0.000000
3,88,Archipiélago de San Andrés,2012,0.000000,0.000000,0.00000,0.000000
4,08,Atlántico,2012,0.000000,0.000000,0.00000,0.000000
...,...,...,...,...,...,...,...
251,70,Sucre,2019,0.000000,0.000000,0.00000,0.000000
252,73,Tolima,2019,104624.420000,76191.120000,82509.22000,1.082924
253,76,Valle del Cauca,2019,53487.550000,38703.780000,40874.10000,1.056075
254,97,Vaupés,2019,0.000000,0.000000,0.00000,0.000000


In [9]:
# Exportar el archivo
output_path = PROJECT_ROOT / "data" / "middle" / "ena_cafe_departamental.csv"
ena_cafe_long.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Archivo guardado en: {output_path}")

Archivo guardado en: /home/ddayann/proyectos/Coffe/proyecto_aplicado_en_analitica_de_datos/data/middle/ena_cafe_departamental.csv


In [10]:
cafe_caldas = ena_cafe[ena_cafe["Departamento"] == "Caldas"].copy()
cafe_caldas

,Codigo,Departamento,Año,Area_plantada,Area_productiva,Produccion,Rendimiento
7,17,Caldas,2012,63153.048130,39557.135528,62517.871130,1.580445
39,17,Caldas,2013,78518.811344,55007.637172,99537.676621,1.809525
71,17,Caldas,2014,80146.239242,57755.254949,103893.906467,1.798865
103,17,Caldas,2015,86804.311378,62553.217490,105878.111170,1.692609
135,17,Caldas,2016,66399.543425,51244.905866,102724.751149,2.004585
167,17,Caldas,2017,59267.670000,42685.940000,96968.580000,2.270000
199,17,Caldas,2018,63942.470000,48614.980000,63660.440000,1.309482
231,17,Caldas,2019,64324.170000,49335.900000,65637.930000,1.330429


In [11]:
# Exportar el archivo
output_path = PROJECT_ROOT / "data" / "middle" / "produccion_caldas_2012-2019.csv"
cafe_caldas.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Archivo guardado en: {output_path}")

Archivo guardado en: /home/ddayann/proyectos/Coffe/proyecto_aplicado_en_analitica_de_datos/data/middle/produccion_caldas_2012-2019.csv
